In [ ]:
# ---------------- Configuration: prediction-head tuning workflow ----------------
from pathlib import Path

RANDOM_STATE = 42
N_SPLITS = 5

ID_COL = 'person_id'
LABEL_COL = 'label'
DAY_COL = 'seq_day'
TOKEN_COL = 'tokens'
GROUP_COL = 'phecode_groups'

# Public/GitHub-safe paths.
# Put de-identified input files under DATA_DIR locally.
# Do not commit institutional EHR data, outputs containing patient-level predictions,
# local manifests, or files with person-level identifiers.
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
OUT_DIR = PROJECT_ROOT / 'outputs' / 'phecode_hybrid_transition_v9_head_tuning'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Original condition source files with ICD/source-code columns.
# Replace these placeholders with local de-identified filenames.
CASE_CONDITION_PATH = DATA_DIR / 'case_condition_source.csv'
CONTROL_CONDITION_PATH = DATA_DIR / 'control_condition_source.csv'

# Optional: if you have a single combined original condition file with an existing label column, set this.
COMBINED_CONDITION_PATH = None

# Official Phecode files saved from the PheWAS package.
PHECODE_MAP_PATH = DATA_DIR / 'phecode_condition_mapping.csv'
PHECODE_INFO_PATH = DATA_DIR / 'phecode_info.csv'

# Optional raw collapsed event-day file from V1, used only as a benchmark if present.
V1_EVENT_DAY_PATH = DATA_DIR / 'raw_condition_event_day.csv'

# Preoperative window relative to surgery.
MIN_DAY = -730
MAX_DAY = 0

# If no exact control filename exists, auto-search likely files inside DATA_DIR only.
AUTO_FIND_SOURCE_FILES = True

# Transition settings.
TRANSITION_GRID = [
    (3, 1),
    (3, 2),
    (7, 2),
    (7, 3),
    (14, 3),
    (14, 7),
    (30, 7),
]
# Primary biologic mode is forward. Bidirectional is a sensitivity/proximity analysis, not causal.
TRANSITION_MODES = ['forward', 'bidirectional']
USE_FIRST_ONSET_FOR_TRANSITIONS = True

# Feature limits.
TOP_K_RAW_FREQ = 1000
TOP_K_HYBRID_FREQ = 1500
TOP_K_PHEGROUP_FREQ = 500
TOP_K_MAPPED_TRANSITION = 2000
TOP_K_DOMAIN_FREQ = 500
TOP_K_DOMAIN_TRANSITION = 500
TOP_K_COMBINED = 3500
TOP_K_UTILIZATION = 50
TOP_K_COMBINED_UTIL = 4000
MIN_PATIENT_FREQ = 5
SPEC_TARGETS = [0.80, 0.85, 0.90, 0.95]

# V9 prediction-head tuning options. Keep this small to avoid overfitting.
RUN_TREE_HEADS = True  # XGBoost/LightGBM are skipped automatically if unavailable.
INNER_CV_SPLITS = 3
HEAD_TUNING_CS = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0]
HEAD_TUNING_L1_RATIOS = [0.05, 0.20, 0.50, 0.80]

print('Output directory name:', OUT_DIR.name)
print('Data directory name:', DATA_DIR.name)
print('Case source file:', CASE_CONDITION_PATH.name)
print('Control source file:', CONTROL_CONDITION_PATH.name)
print('Phecode map file:', PHECODE_MAP_PATH.name)
print('Phecode info file:', PHECODE_INFO_PATH.name)


## Public release privacy checklist

Before committing this notebook to GitHub:

- Clear all cell outputs.
- Do not commit local `data/` files or generated `outputs/`.
- Do not commit CSVs containing `person_id`, patient-level predictions, source condition records, dates, or institutional file paths.
- Keep only de-identified example schemas, README text, and code.
- Use repo-relative paths (`data/`, `outputs/`) instead of local absolute paths.


In [ ]:
# ---------------- Imports ----------------
import ast
import json
import math
import re
import warnings
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.base import clone
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
    auc,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)
from sklearn.model_selection import GridSearchCV


In [ ]:
# ---------------- Source-file discovery and basic utilities ----------------
SOURCE_REQUIRED_COLS = [
    ID_COL,
    'condition_source_value',
    'source_code_clean',
    'days_from_surgery',
    'condition_label',
    'condition_token',
    'leakage_flag',
]

BAD_SOURCE_FILE_REGEX = re.compile(
    r'(metrics|prediction|coefficient|importance|manifest|summary|audit|transition|frequency|features|oof|phecode|event_day|mapped)',
    re.I,
)


def norm_text(x):
    if pd.isna(x):
        return ''
    return re.sub(r'[^a-z0-9]+', '_', str(x).strip().lower()).strip('_')


def parse_token_payload(x):
    if isinstance(x, (list, tuple, set, np.ndarray)):
        vals = list(x)
    elif pd.isna(x):
        vals = []
    else:
        s = str(x).strip()
        vals = None
        if s.startswith('[') and s.endswith(']'):
            try:
                parsed = ast.literal_eval(s)
                vals = parsed if isinstance(parsed, (list, tuple, set)) else [parsed]
            except Exception:
                vals = None
        if vals is None:
            if '|' in s:
                vals = s.split('|')
            elif ';' in s:
                vals = s.split(';')
            else:
                vals = [s]
    return sorted(set(str(v).strip() for v in vals if str(v).strip() and str(v).strip().lower() != 'nan'))


def score_original_condition_file(path):
    try:
        tmp = pd.read_csv(path, nrows=10, low_memory=False)
        tmp.columns = [str(c).strip() for c in tmp.columns]
    except Exception:
        return -999, []
    cols_lower = {c.lower(): c for c in tmp.columns}
    score = 0
    hits = []
    for c in SOURCE_REQUIRED_COLS:
        if c.lower() in cols_lower:
            score += 10
            hits.append(cols_lower[c.lower()])
    if BAD_SOURCE_FILE_REGEX.search(path.name):
        score -= 30
    if 'condition_occurrence_id' in cols_lower:
        score += 5
    return score, hits


def find_source_files():
    case_path = Path(CASE_CONDITION_PATH) if CASE_CONDITION_PATH is not None else None
    control_path = Path(CONTROL_CONDITION_PATH) if CONTROL_CONDITION_PATH is not None else None

    if COMBINED_CONDITION_PATH is not None and Path(COMBINED_CONDITION_PATH).exists():
        return {'combined': Path(COMBINED_CONDITION_PATH)}

    if not AUTO_FIND_SOURCE_FILES:
        return {'case': case_path, 'control': control_path}

    patterns = [
        '*condition*cleaned*names*leakage*flags*.csv',
        '*recurrence*condition*.csv',
        '*control*condition*.csv',
        '*condition*pre*.csv',
    ]
    rows = []
    seen = set()
    for pattern in patterns:
        for p in DATA_DIR.rglob(pattern):
            if p in seen:
                continue
            seen.add(p)
            s, hits = score_original_condition_file(p)
            if s > 0:
                rows.append({'score': s, 'file': p, 'hits': hits, 'name': p.name.lower()})
    cand = pd.DataFrame(rows).sort_values(['score', 'name'], ascending=[False, True]) if rows else pd.DataFrame()
    if len(cand):
        display(cand.assign(file=cand['file'].map(lambda p: Path(p).name))[['score', 'file', 'hits']].head(20))

    if case_path is None or not case_path.exists():
        case_matches = cand[cand['name'].str.contains('recurrence|case', regex=True)] if len(cand) else pd.DataFrame()
        if len(case_matches):
            case_path = case_matches.iloc[0]['file']
    if control_path is None or not control_path.exists():
        control_matches = cand[cand['name'].str.contains('control|nonrecurrence|non_recurrence|no_recurrence', regex=True)] if len(cand) else pd.DataFrame()
        if len(control_matches):
            control_path = control_matches.iloc[0]['file']

    return {'case': case_path, 'control': control_path}


source_paths = find_source_files()
print('Source paths detected:')
for k, p in source_paths.items():
    print(k, Path(p).name if p is not None else None, 'exists=', Path(p).exists() if p is not None else None)


In [ ]:
# ---------------- ICD-to-Phecode mapping helpers ----------------
def infer_source_vocab(x):
    # Infer ICD9CM/ICD10CM from condition_source_value prefix.
    if pd.isna(x):
        return np.nan
    s = str(x).strip().upper()
    if s.startswith(('I10:', 'I0:', '10:', 'ICD10:', 'ICD10CM:')):
        return 'ICD10CM'
    if s.startswith(('I9:', '9:', 'ICD9:', 'ICD9CM:')):
        return 'ICD9CM'
    return np.nan


def clean_source_code(x):
    # Extract code after ICD prefix and normalize basic formatting.
    if pd.isna(x):
        return np.nan
    s = str(x).strip().upper()
    s = re.sub(r'^(ICD10CM|ICD10|I10|I0|10|ICD9CM|ICD9|I9|9)\s*[:\-]\s*', '', s)
    s = s.replace(' ', '')
    return s if s else np.nan


def code_variants(code):
    # Generate variants for ICD decimal-format matching.
    if pd.isna(code):
        return []
    s = str(code).strip().upper()
    if not s or s in {'NAN', 'NONE'}:
        return []
    variants = {s}
    if '.' in s:
        left, right = s.split('.', 1)
        stripped = right.rstrip('0')
        variants.add(left + ('.' + stripped if stripped else ''))
        if len(right) == 1:
            variants.add(left + '.' + right + '0')
    variants.add(s.replace('.', ''))
    return sorted(v for v in variants if v)


def load_phecode_maps(phecode_map_path=PHECODE_MAP_PATH, phecode_info_path=PHECODE_INFO_PATH):
    if not Path(phecode_map_path).exists():
        raise FileNotFoundError(f'Missing Phecode map: {phecode_map_path}')
    if not Path(phecode_info_path).exists():
        raise FileNotFoundError(f'Missing Phecode info: {phecode_info_path}')

    phemap = pd.read_csv(phecode_map_path, dtype=str, low_memory=False)
    phemap.columns = [str(c).strip() for c in phemap.columns]
    required = {'vocabulary_id', 'code', 'phecode'}
    missing = required - set(phemap.columns)
    if missing:
        raise ValueError(f'Phecode map missing required columns: {missing}')

    phemap['vocabulary_id'] = phemap['vocabulary_id'].astype(str).str.upper().str.strip()
    phemap['code'] = phemap['code'].astype(str).str.upper().str.strip()
    phemap['phecode'] = phemap['phecode'].astype(str).str.strip()

    pheinfo = pd.read_csv(phecode_info_path, dtype=str, low_memory=False)
    pheinfo.columns = [str(c).strip() for c in pheinfo.columns]
    pheinfo['phecode'] = pheinfo['phecode'].astype(str).str.strip()

    keep_info = [c for c in ['phecode', 'description', 'group', 'groupnum'] if c in pheinfo.columns]
    phemap = phemap.merge(pheinfo[keep_info].drop_duplicates('phecode'), on='phecode', how='left')
    return phemap


phemap = load_phecode_maps()
print('Phecode map rows:', phemap.shape)
display(phemap.head())


In [ ]:
# ---------------- Read original condition files and apply source-code mapping ----------------
def read_condition_source(path, label=None, cohort_name=None):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Missing source file: {path}')
    df = pd.read_csv(path, low_memory=False)
    df.columns = [str(c).strip() for c in df.columns]
    if label is not None:
        df[LABEL_COL] = int(label)
    elif LABEL_COL not in df.columns:
        raise ValueError(f'{path} has no label column and no label was supplied.')
    if cohort_name is not None:
        df['cohort_name'] = cohort_name
    elif 'cohort_name' not in df.columns:
        df['cohort_name'] = df[LABEL_COL].map({1: 'case', 0: 'control'}).fillna('unknown')
    df['source_file'] = str(path)
    return df


def prepare_condition_source(df):
    df = df.copy()
    required = [ID_COL, 'condition_source_value', 'days_from_surgery']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'Missing required columns: {missing}')

    df[ID_COL] = pd.to_numeric(df[ID_COL], errors='coerce')
    df['days_from_surgery'] = pd.to_numeric(df['days_from_surgery'], errors='coerce')

    print('Rows before required-field filter:', df.shape)
    print('Patients before required-field filter:', df[ID_COL].nunique(dropna=True))

    df = df.dropna(subset=[ID_COL, 'days_from_surgery', 'condition_source_value']).copy()
    df[ID_COL] = df[ID_COL].astype(int)
    df[DAY_COL] = df['days_from_surgery'].astype(int)

    print('Rows after required-field filter:', df.shape)
    print('Patients after required-field filter:', df[ID_COL].nunique())
    print('days_from_surgery summary before window filter:')
    display(df[DAY_COL].describe().to_frame().T)

    before_window = len(df)
    df = df[(df[DAY_COL] >= MIN_DAY) & (df[DAY_COL] <= MAX_DAY)].copy()
    print(f'Rows removed outside [{MIN_DAY}, {MAX_DAY}] window:', before_window - len(df))
    print('Rows after window filter:', df.shape)
    print('Patients after window filter:', df[ID_COL].nunique())

    if 'leakage_flag' in df.columns:
        # Important: do not require exact equality to 'not_explicit_cancer'.
        # Some source files contain alternative non-leakage labels or missing flags.
        # Remove only rows explicitly flagged as cancer leakage, while preserving
        # 'not_explicit_cancer', blank/missing, and other non-cancer flags.
        flag = df['leakage_flag'].astype(str).str.strip().str.lower()
        if len(df) > 0:
            print('Top leakage_flag values after window filter:')
            display(flag.value_counts(dropna=False).head(20).rename_axis('leakage_flag').reset_index(name='n_rows'))

        is_not_explicit = flag.str.contains(r'not[_\s-]*explicit[_\s-]*cancer', regex=True, na=False)
        is_explicit = flag.str.contains(r'explicit[_\s-]*cancer|pancreatic[_\s-]*cancer|pdac|malignant|metasta', regex=True, na=False)
        remove_leakage = is_explicit & ~is_not_explicit

        before_leak = len(df)
        df = df[~remove_leakage].copy()
        print('Removed explicit cancer leakage rows:', before_leak - len(df))
        print('Rows after leakage filter:', df.shape)
        print('Patients after leakage filter:', df[ID_COL].nunique())

    df['_row_id'] = np.arange(len(df), dtype=int)
    df['source_vocabulary_id'] = df['condition_source_value'].apply(infer_source_vocab)

    # Prefer source_code_clean when available, but always strip ICD prefixes.
    # Some combined files store values like I0:E78.5, I9:401.9, or ICD9CM:401.9
    # inside source_code_clean, so using it directly prevents Phecode matching.
    if 'source_code_clean' in df.columns:
        source_code_raw = df['source_code_clean'].where(df['source_code_clean'].notna(), df['condition_source_value'])
    else:
        source_code_raw = df['condition_source_value']

    df['source_code_base'] = source_code_raw.apply(clean_source_code)
    df['source_code_base'] = df['source_code_base'].astype(str).str.upper().str.strip()
    df.loc[df['source_code_base'].isin(['', 'NAN', 'NONE']), 'source_code_base'] = np.nan

    if 'condition_token' in df.columns:
        fallback_core = df['condition_token'].astype(str).map(norm_text)
    elif 'condition_label' in df.columns:
        fallback_core = df['condition_label'].astype(str).map(norm_text)
    else:
        fallback_core = df['condition_source_value'].astype(str).map(norm_text)
    fallback_core = fallback_core.replace('', np.nan).fillna('unknown_condition')
    df['fallback_token'] = 'unmapped_condition::' + fallback_core
    return df


def map_prepared_conditions_to_phecodes(prepared_df, phemap):
    base = prepared_df.copy()
    base['code_variant'] = base['source_code_base'].apply(code_variants)
    expanded = base.explode('code_variant').dropna(subset=['source_vocabulary_id', 'code_variant']).copy()
    expanded['code_variant'] = expanded['code_variant'].astype(str).str.upper().str.strip()

    mapped = expanded.merge(
        phemap,
        left_on=['source_vocabulary_id', 'code_variant'],
        right_on=['vocabulary_id', 'code'],
        how='left',
    )

    matched = mapped[mapped['phecode'].notna()].copy()
    matched = matched.drop_duplicates(['_row_id', 'phecode'])

    matched_row_ids = set(matched['_row_id'].astype(int))
    fallback = base[~base['_row_id'].isin(matched_row_ids)].copy()
    fallback['code_variant'] = np.nan
    fallback['vocabulary_id'] = np.nan
    fallback['code'] = np.nan
    fallback['phecode'] = np.nan
    fallback['description'] = np.nan
    fallback['group'] = np.nan
    fallback['groupnum'] = np.nan

    token_df = pd.concat([matched, fallback], ignore_index=True, sort=False)

    token_df['mapped_phecode_token'] = (
        'phecode::'
        + token_df['phecode'].astype(str)
        + '::'
        + token_df['description'].fillna('unknown').astype(str)
    )
    token_df['analysis_token'] = np.where(
        token_df['phecode'].notna(),
        token_df['mapped_phecode_token'],
        token_df['fallback_token'],
    )
    token_df['analysis_group_token'] = np.where(
        token_df['phecode'].notna(),
        'phegroup::' + token_df['group'].fillna('unknown').astype(str),
        'phegroup::unmapped',
    )

    # Mapped-only table is used for biologically interpretable transitions.
    # It must be constructed from matched rows, so create the same token columns here too.
    mapped_only = matched.copy()
    mapped_only['mapped_phecode_token'] = (
        'phecode::'
        + mapped_only['phecode'].astype(str)
        + '::'
        + mapped_only['description'].fillna('unknown').astype(str)
    )
    mapped_only['analysis_token'] = mapped_only['mapped_phecode_token']
    mapped_only['analysis_group_token'] = 'phegroup::' + mapped_only['group'].fillna('unknown').astype(str)

    return token_df, mapped_only, expanded, mapped


if 'combined' in source_paths:
    raw_both = read_condition_source(source_paths['combined'])
else:
    case_df = read_condition_source(source_paths['case'], label=1, cohort_name='case')
    control_df = read_condition_source(source_paths['control'], label=0, cohort_name='control')
    raw_both = pd.concat([case_df, control_df], ignore_index=True, sort=False)

print('Raw combined condition rows:', raw_both.shape)
print('Raw combined patients:', raw_both[ID_COL].nunique())
display(raw_both[[ID_COL, LABEL_COL]].drop_duplicates()[LABEL_COL].value_counts().rename_axis('label').reset_index(name='n_patients'))

prepared = prepare_condition_source(raw_both)
print('After filters:', prepared.shape)
print('Patients after filters:', prepared[ID_COL].nunique())

# V7 source-cohort anchor: this is the analytic cohort after required-field, window,
# and explicit-leakage filtering. Later cells must preserve these patients even if
# mapped-only/transition features are absent.
SOURCE_PATIENT_IDS_AFTER_FILTERS = sorted(prepared[ID_COL].dropna().astype(int).unique().tolist())
SOURCE_N_AFTER_FILTERS = len(SOURCE_PATIENT_IDS_AFTER_FILTERS)
print('V7 source-cohort anchor after filters:', SOURCE_N_AFTER_FILTERS, 'patients')

token_df, mapped_only_df, expanded_df, raw_mapped_attempt_df = map_prepared_conditions_to_phecodes(prepared, phemap)

print('Rows before expansion:', prepared.shape)
print('Rows after expansion:', expanded_df.shape)
print('Matched official Phecode rows:', mapped_only_df.shape)
print('Hybrid token rows after adding unmapped fallback:', token_df.shape)
print('Patients retained in hybrid token rows:', token_df[ID_COL].nunique())
print('Patients with official Phecode match:', mapped_only_df[ID_COL].nunique())

display(token_df[[ID_COL, LABEL_COL, 'cohort_name', 'condition_source_value', 'source_vocabulary_id', 'source_code_base', 'code_variant', 'condition_label', 'phecode', 'description', 'group', 'analysis_token']].head(30))


In [ ]:
# ---------------- Mapping-retention audit ----------------
def mapping_retention_audit(prepared_df, mapped_only_df, token_df):
    base = prepared_df[[ID_COL, LABEL_COL, 'cohort_name']].drop_duplicates(ID_COL).copy()
    matched_ids = set(mapped_only_df[ID_COL].astype(int))
    hybrid_ids = set(token_df[ID_COL].astype(int))
    base['has_phecode_match'] = base[ID_COL].isin(matched_ids)
    base['retained_in_hybrid'] = base[ID_COL].isin(hybrid_ids)

    print('Patient-level retention:')
    display(
        base.groupby(['cohort_name', LABEL_COL, 'has_phecode_match', 'retained_in_hybrid'])
        .size()
        .reset_index(name='n_patients')
    )

    print('Row-level mapping status:')
    row_status = prepared_df[['_row_id', ID_COL, LABEL_COL, 'cohort_name']].copy()
    row_status['has_phecode_match'] = row_status['_row_id'].isin(set(mapped_only_df['_row_id'].astype(int)))
    display(
        row_status.groupby(['cohort_name', LABEL_COL, 'has_phecode_match'])
        .size()
        .reset_index(name='n_condition_rows')
    )

    print('Source vocabulary parse failures:')
    display(
        prepared_df.assign(
            missing_vocab=prepared_df['source_vocabulary_id'].isna(),
            missing_code=prepared_df['source_code_base'].isna(),
        )
        .groupby(['missing_vocab', 'missing_code'])
        .size()
        .reset_index(name='n_rows')
    )

    print('Top condition_source_value prefixes:')
    display(
        prepared_df['condition_source_value']
        .astype(str)
        .str.extract(r'^([^:]+):')[0]
        .fillna('no_prefix')
        .value_counts()
        .head(20)
        .rename_axis('prefix')
        .reset_index(name='n_rows')
    )
    return base, row_status

patient_mapping_audit, row_mapping_audit = mapping_retention_audit(prepared, mapped_only_df, token_df)

patient_mapping_audit.to_csv(OUT_DIR / 'v3_patient_mapping_retention_audit.csv', index=False)
row_mapping_audit.to_csv(OUT_DIR / 'v3_row_mapping_retention_audit.csv', index=False)
raw_mapped_attempt_df.to_csv(OUT_DIR / 'v3_raw_icd_to_phecode_mapping_attempts.csv', index=False)
token_df.to_csv(OUT_DIR / 'v3_hybrid_long_token_table.csv', index=False)
mapped_only_df.to_csv(OUT_DIR / 'v3_mapped_phecode_long_token_table.csv', index=False)
print('Saved mapping audit files to:', OUT_DIR)


In [ ]:
# ---------------- Collapse long token rows into event-day tables ----------------
def collapse_event_days(long_df, token_col='analysis_token', group_col='analysis_group_token'):
    df = long_df.copy()
    df = df[df[token_col].notna()].copy()
    df[token_col] = df[token_col].astype(str)
    df[group_col] = df[group_col].astype(str)

    agg = {
        TOKEN_COL: (token_col, lambda x: sorted(set(str(v) for v in x if str(v) and str(v).lower() != 'nan'))),
        GROUP_COL: (group_col, lambda x: sorted(set(str(v) for v in x if str(v) and str(v).lower() != 'nan'))),
    }
    if 'condition_label' in df.columns:
        agg['raw_conditions'] = ('condition_label', lambda x: sorted(set(str(v) for v in x if pd.notna(v))))

    out = df.groupby([ID_COL, DAY_COL], as_index=False).agg(**agg)
    out = out.merge(df[[ID_COL, LABEL_COL, 'cohort_name']].drop_duplicates(ID_COL), on=ID_COL, how='left')
    return out


hybrid_condition_days = collapse_event_days(token_df)
mapped_phecode_days = collapse_event_days(mapped_only_df)

# V7 hard check: collapsing tokens must not reduce the post-filter source cohort.
if 'SOURCE_PATIENT_IDS_AFTER_FILTERS' not in globals():
    raise RuntimeError('SOURCE_PATIENT_IDS_AFTER_FILTERS is missing. Restart kernel and Run All from the top.')
_source_ids = set(SOURCE_PATIENT_IDS_AFTER_FILTERS)
_hybrid_ids = set(hybrid_condition_days[ID_COL].dropna().astype(int))
if _hybrid_ids != _source_ids:
    missing = sorted(_source_ids - _hybrid_ids)[:20]
    extra = sorted(_hybrid_ids - _source_ids)[:20]
    raise AssertionError(
        f'Hybrid event-day table has {len(_hybrid_ids)} patients but source cohort has {len(_source_ids)}. '
        f'Missing example IDs: {missing}; extra example IDs: {extra}. '
        'Do not proceed; token/event-day collapse is dropping patients.'
    )
print('V7 source-cohort collapse check passed:', len(_hybrid_ids), 'patients')

hybrid_path = OUT_DIR / 'pdac_case_control_phecode_hybrid_condition_event_days.csv'
mapped_path = OUT_DIR / 'pdac_case_control_mapped_phecode_condition_event_days.csv'
hybrid_condition_days.to_csv(hybrid_path, index=False)
mapped_phecode_days.to_csv(mapped_path, index=False)

print('Hybrid event-day table:', hybrid_condition_days.shape, 'patients:', hybrid_condition_days[ID_COL].nunique())
display(hybrid_condition_days[[ID_COL, LABEL_COL]].drop_duplicates()[LABEL_COL].value_counts().rename_axis('label').reset_index(name='n_patients'))
display(hybrid_condition_days.head())

print('Mapped-only event-day table:', mapped_phecode_days.shape, 'patients:', mapped_phecode_days[ID_COL].nunique())
display(mapped_phecode_days[[ID_COL, LABEL_COL]].drop_duplicates()[LABEL_COL].value_counts().rename_axis('label').reset_index(name='n_patients'))
display(mapped_phecode_days.head())

print('Saved:', hybrid_path)
print('Saved:', mapped_path)


In [ ]:
# ---------------- Optional: load raw condition event-day baseline if present ----------------
def load_v1_event_day(path):
    path = Path(path)
    if not path.exists():
        print('V1 event-day file not found; skipping V1 raw baseline:', path)
        return pd.DataFrame(columns=[ID_COL, DAY_COL, TOKEN_COL, LABEL_COL])
    df = pd.read_csv(path, low_memory=False)
    df.columns = [str(c).strip() for c in df.columns]
    lower = {c.lower(): c for c in df.columns}
    day_col = next((lower[c] for c in ['seq_day', 'event_day', 'days_from_surgery', 'day'] if c in lower), None)
    tok_col = next((lower[c] for c in ['tokens', 'condition_token', 'condition', 'condition_name', 'concept_name'] if c in lower), None)
    if day_col is None or tok_col is None:
        print('V1 event-day file missing day/token columns; skipping.')
        return pd.DataFrame(columns=[ID_COL, DAY_COL, TOKEN_COL, LABEL_COL])
    out = df.rename(columns={day_col: DAY_COL, tok_col: TOKEN_COL}).copy()
    out[ID_COL] = pd.to_numeric(out[ID_COL], errors='coerce')
    out[DAY_COL] = pd.to_numeric(out[DAY_COL], errors='coerce')
    out = out.dropna(subset=[ID_COL, DAY_COL]).copy()
    out[ID_COL] = out[ID_COL].astype(int)
    out[DAY_COL] = out[DAY_COL].astype(int)
    out[TOKEN_COL] = out[TOKEN_COL].apply(parse_token_payload)
    out = out[out[TOKEN_COL].map(len) > 0].copy()
    out = out.groupby([ID_COL, DAY_COL], as_index=False)[TOKEN_COL].agg(lambda xs: sorted(set(t for sub in xs for t in sub)))
    if LABEL_COL in df.columns:
        out = out.merge(df[[ID_COL, LABEL_COL]].dropna().drop_duplicates(ID_COL), on=ID_COL, how='left')
    return out


v1_condition_days = load_v1_event_day(V1_EVENT_DAY_PATH)
print('V1 condition days:', v1_condition_days.shape, 'patients:', v1_condition_days[ID_COL].nunique())


In [ ]:
# ---------------- Outcome table and sequence construction: preserve full hybrid cohort ----------------

def audit_event_day_sources_for_y():
    print('Event-day source patient counts before y_df construction:')
    for name in ['hybrid_condition_days', 'mapped_phecode_days', 'v1_condition_days']:
        if name in globals():
            df = globals()[name]
            if isinstance(df, pd.DataFrame) and ID_COL in df.columns:
                print(name, df.shape, 'patients:', df[ID_COL].nunique())
                if LABEL_COL in df.columns and len(df):
                    display(
                        df[[ID_COL, LABEL_COL]]
                        .drop_duplicates(ID_COL)[LABEL_COL]
                        .value_counts()
                        .sort_index()
                        .rename_axis('label')
                        .reset_index(name='n_patients')
                    )

def build_y_df_from_hybrid(hybrid_df):
    if hybrid_df is None or not len(hybrid_df):
        raise ValueError('hybrid_condition_days is empty; cannot define analytic cohort.')
    required = [ID_COL, LABEL_COL]
    missing = [c for c in required if c not in hybrid_df.columns]
    if missing:
        raise ValueError(f'hybrid_condition_days missing required columns: {missing}')

    cols = [ID_COL, LABEL_COL]
    if 'cohort_name' in hybrid_df.columns:
        cols.append('cohort_name')

    y = hybrid_df[cols].drop_duplicates(ID_COL).copy()
    y[ID_COL] = pd.to_numeric(y[ID_COL], errors='coerce')
    y[LABEL_COL] = pd.to_numeric(y[LABEL_COL], errors='coerce')
    y = y.dropna(subset=[ID_COL, LABEL_COL]).copy()
    y[ID_COL] = y[ID_COL].astype(int)
    y[LABEL_COL] = y[LABEL_COL].astype(int)
    if 'cohort_name' not in y.columns:
        y['cohort_name'] = y[LABEL_COL].map({1: 'case', 0: 'control'}).fillna('unknown')
    y = y.sort_values(ID_COL).reset_index(drop=True)
    return y


def build_sequence_map(df, token_col=TOKEN_COL, patient_ids=None):
    seq = {}
    if df is not None and len(df):
        for pid, g in df.sort_values([ID_COL, DAY_COL]).groupby(ID_COL):
            seq[int(pid)] = [(int(r[DAY_COL]), list(r[token_col])) for _, r in g.iterrows()]
    if patient_ids is not None:
        for pid in patient_ids:
            seq.setdefault(int(pid), [])
    return seq


audit_event_day_sources_for_y()
y_df = build_y_df_from_hybrid(hybrid_condition_days)
all_patient_ids = y_df[ID_COL].astype(int).tolist()
base_patient_set = set(all_patient_ids)

# Build sequence maps, explicitly adding empty sequences for patients absent from narrower tables.
hybrid_seq = build_sequence_map(hybrid_condition_days, TOKEN_COL, all_patient_ids)
mapped_seq = build_sequence_map(mapped_phecode_days, TOKEN_COL, all_patient_ids)
group_seq = build_sequence_map(mapped_phecode_days, GROUP_COL, all_patient_ids)
v1_raw_seq = build_sequence_map(v1_condition_days, TOKEN_COL, all_patient_ids)

print('Outcome table y_df:', y_df.shape, 'patients:', y_df[ID_COL].nunique())
print('Case rate:', y_df[LABEL_COL].mean())
display(
    y_df[LABEL_COL]
    .value_counts()
    .sort_index()
    .rename_axis('label')
    .reset_index(name='n_patients')
)

# Hard guard against accidental cohort shrinkage.
expected_n = hybrid_condition_days[ID_COL].nunique()
actual_n = y_df[ID_COL].nunique()
if actual_n != expected_n:
    raise AssertionError(
        f'y_df patient count ({actual_n}) does not match hybrid_condition_days ({expected_n}). '
        'Do not let transition/mapped-only tables define the cohort.'
    )


# V7 helper: make resetting the analytic cohort explicit and reusable.
def reset_cohort_from_hybrid_event_days(verbose=True):
    """Rebuild y_df, all_patient_ids, and all sequence maps from hybrid_condition_days.

    This must be called before feature construction and before model fitting. It prevents
    stale notebook state from carrying forward an older 505-patient y_df after the
    event-day table has been rebuilt to 715 patients.
    """
    global y_df, all_patient_ids, base_patient_set
    global hybrid_seq, mapped_seq, group_seq, v1_raw_seq, domain_seq
    global mapped_transition_seq, group_transition_seq, domain_transition_seq

    y_df = build_y_df_from_hybrid(hybrid_condition_days)
    all_patient_ids = y_df[ID_COL].astype(int).tolist()
    base_patient_set = set(all_patient_ids)

    hybrid_seq = build_sequence_map(hybrid_condition_days, TOKEN_COL, all_patient_ids)
    mapped_seq = build_sequence_map(mapped_phecode_days, TOKEN_COL, all_patient_ids)
    group_seq = build_sequence_map(mapped_phecode_days, GROUP_COL, all_patient_ids)
    v1_raw_seq = build_sequence_map(v1_condition_days, TOKEN_COL, all_patient_ids)

    # domain_seq may not exist yet when this is first called; it is rebuilt after domain_days is created.
    if 'domain_days' in globals() and isinstance(domain_days, pd.DataFrame):
        domain_seq = build_sequence_map(domain_days, TOKEN_COL, all_patient_ids)
    else:
        domain_seq = {int(pid): [] for pid in all_patient_ids}

    mapped_transition_seq = first_onset_sequence(mapped_seq) if 'first_onset_sequence' in globals() and USE_FIRST_ONSET_FOR_TRANSITIONS else mapped_seq
    group_transition_seq = first_onset_sequence(group_seq) if 'first_onset_sequence' in globals() and USE_FIRST_ONSET_FOR_TRANSITIONS else group_seq
    domain_transition_seq = first_onset_sequence(domain_seq) if 'first_onset_sequence' in globals() and USE_FIRST_ONSET_FOR_TRANSITIONS else domain_seq

    expected_n = hybrid_condition_days[ID_COL].nunique()
    actual_n = y_df[ID_COL].nunique()
    if actual_n != expected_n:
        raise AssertionError(
            f'V7 cohort reset failed: y_df patient count ({actual_n}) does not match '
            f'hybrid_condition_days ({expected_n}).'
        )
    if verbose:
        print('V7 cohort reset complete.')
        print('hybrid_condition_days patients:', expected_n)
        print('y_df patients:', actual_n)
        display(
            y_df[LABEL_COL]
            .value_counts()
            .sort_index()
            .rename_axis('label')
            .reset_index(name='n_patients')
        )
    return y_df


In [ ]:
# ---------------- PDAC-relevant domain layer from Phecode labels/groups ----------------
# This is an interpretable secondary layer. It is not required for the official Phecode category transition models.
DOMAIN_RULES = {
    'metabolic_dysfunction': [
        r'diabetes', r'obesity', r'lipoid', r'hypercholesterol', r'hyperlipidemia', r'endocrine/metabolic'
    ],
    'biliary_hepatic_dysfunction': [
        r'bile', r'biliary', r'cholang', r'jaundice', r'liver', r'hepatic', r'pancrea', r'digestive'
    ],
    'infection_inflammation': [
        r'infection', r'sepsis', r'abscess', r'cellulitis', r'pneumonia', r'osteomyelitis', r'inflammatory'
    ],
    'nutritional_symptom_burden': [
        r'weight loss', r'malnutrition', r'cache', r'nausea', r'vomit', r'abdominal pain', r'appetite', r'symptoms'
    ],
    'thromboembolism_vascular': [
        r'thrombo', r'embol', r'venous', r'coag', r'infarction', r'ischemic'
    ],
    'renal_dysfunction': [
        r'renal', r'kidney', r'neph', r'genitourinary'
    ],
    'cardiopulmonary_frailty': [
        r'heart failure', r'coronary', r'atherosclerosis', r'cardiac', r'pulmonary', r'copd', r'respiratory', r'circulatory system'
    ],
    'musculoskeletal_frailty': [
        r'osteoporosis', r'fracture', r'weakness', r'debility', r'musculoskeletal'
    ],
}

PLAUSIBLE_DOMAIN_TRANSITIONS = {
    'metabolic_dysfunction': ['renal_dysfunction', 'cardiopulmonary_frailty', 'infection_inflammation'],
    'biliary_hepatic_dysfunction': ['infection_inflammation', 'nutritional_symptom_burden'],
    'infection_inflammation': ['renal_dysfunction', 'nutritional_symptom_burden', 'cardiopulmonary_frailty'],
    'nutritional_symptom_burden': ['musculoskeletal_frailty', 'cardiopulmonary_frailty'],
    'thromboembolism_vascular': ['cardiopulmonary_frailty', 'renal_dysfunction'],
    'renal_dysfunction': ['cardiopulmonary_frailty'],
}


def token_to_domain(token):
    s = str(token).lower()
    domains = []
    for domain, pats in DOMAIN_RULES.items():
        if any(re.search(p, s, re.I) for p in pats):
            domains.append('domain::' + domain)
    return sorted(set(domains))


def build_domain_event_days_from_mapped(mapped_days):
    rows = []
    for _, r in mapped_days.iterrows():
        doms = []
        for t in r[TOKEN_COL]:
            doms.extend(token_to_domain(t))
        doms = sorted(set(doms))
        if doms:
            rows.append({
                ID_COL: int(r[ID_COL]),
                DAY_COL: int(r[DAY_COL]),
                TOKEN_COL: doms,
                LABEL_COL: int(r[LABEL_COL]),
                'cohort_name': r.get('cohort_name', ''),
            })
    return pd.DataFrame(rows)


domain_days = build_domain_event_days_from_mapped(mapped_phecode_days)
# V7: rebuild domain sequence using the current full cohort.
domain_seq = build_sequence_map(domain_days, TOKEN_COL, all_patient_ids)
print('Domain event-day table:', domain_days.shape, 'patients:', domain_days[ID_COL].nunique() if len(domain_days) else 0)
display(domain_days.head())


In [ ]:
# ---------------- First-onset transformation and visual audit ----------------
def first_onset_sequence(seq_map):
    out = {}
    for pid, seq in seq_map.items():
        first = {}
        for day, toks in sorted(seq, key=lambda x: x[0]):
            for t in toks:
                if t not in first:
                    first[t] = int(day)
        by_day = defaultdict(list)
        for t, d in first.items():
            by_day[int(d)].append(t)
        out[int(pid)] = [(d, sorted(ts)) for d, ts in sorted(by_day.items())]
    return out


mapped_transition_seq = first_onset_sequence(mapped_seq) if USE_FIRST_ONSET_FOR_TRANSITIONS else mapped_seq
group_transition_seq = first_onset_sequence(group_seq) if USE_FIRST_ONSET_FOR_TRANSITIONS else group_seq
domain_transition_seq = first_onset_sequence(domain_seq) if USE_FIRST_ONSET_FOR_TRANSITIONS else domain_seq

# Quick burden summaries.
def patient_burden_from_days(df, name):
    if df is None or not len(df):
        return pd.DataFrame()
    tmp = df.copy()
    tmp['n_tokens_day'] = tmp[TOKEN_COL].map(len)
    burden = (
        tmp.groupby([ID_COL, LABEL_COL], as_index=False)
        .agg(
            n_event_days=(DAY_COL, 'nunique'),
            total_tokens=('n_tokens_day', 'sum'),
            first_day=(DAY_COL, 'min'),
            last_day=(DAY_COL, 'max'),
        )
    )
    burden['source'] = name
    return burden

burden_df = pd.concat([
    patient_burden_from_days(hybrid_condition_days, 'hybrid'),
    patient_burden_from_days(mapped_phecode_days, 'mapped_phecode'),
    patient_burden_from_days(domain_days, 'domain'),
], ignore_index=True)

print('Patient burden summary:')
display(burden_df.groupby(['source', LABEL_COL])[['n_event_days', 'total_tokens', 'first_day', 'last_day']].describe().round(2))

plt.figure(figsize=(8, 5))
for lab, label_name in [(0, 'Control'), (1, 'Case')]:
    vals = burden_df[(burden_df['source'] == 'hybrid') & (burden_df[LABEL_COL] == lab)]['total_tokens']
    plt.hist(vals, bins=30, alpha=0.55, label=label_name)
plt.xlabel('Hybrid condition documentation-token burden per patient')
plt.ylabel('Number of patients')
plt.title('Hybrid condition documentation burden by outcome')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
for lab, label_name in [(0, 'Control'), (1, 'Case')]:
    vals = hybrid_condition_days[hybrid_condition_days[LABEL_COL] == lab][DAY_COL]
    plt.hist(vals, bins=50, alpha=0.55, label=label_name)
plt.axvline(0, linestyle='--', linewidth=1)
plt.xlabel('Days relative to surgery')
plt.ylabel('Condition documentation event-days')
plt.title('Timing distribution of hybrid condition documentation before surgery')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ---------------- Utilization / documentation-intensity adjustment features ----------------
def build_utilization_features(event_days, token_col=TOKEN_COL, patient_ids=None):
    """Patient-level EHR condition-documentation intensity features.

    These features are not intended as biology. They quantify how much condition documentation
    exists per patient and when it occurs relative to surgery, so downstream models can test
    whether condition/category transitions add signal beyond observation intensity.
    """
    if event_days is None or not len(event_days):
        return pd.DataFrame({ID_COL: patient_ids or []})

    df = event_days.copy()
    df['n_tokens_day'] = df[token_col].map(len)

    util = (
        df.groupby([ID_COL, LABEL_COL, 'cohort_name'], as_index=False)
        .agg(
            n_condition_event_days=(DAY_COL, 'nunique'),
            total_condition_tokens=('n_tokens_day', 'sum'),
            first_condition_day=(DAY_COL, 'min'),
            last_condition_day=(DAY_COL, 'max'),
        )
    )

    unique_tokens = (
        df[[ID_COL, token_col]]
        .explode(token_col)
        .dropna(subset=[token_col])
        .groupby(ID_COL)[token_col]
        .nunique()
        .reset_index(name='n_unique_condition_tokens')
    )

    def recent_count(window):
        return (
            df[df[DAY_COL] >= -int(window)]
            .groupby(ID_COL)[DAY_COL]
            .nunique()
            .reset_index(name=f'n_recent_condition_event_days_{int(window)}d')
        )

    util = util.merge(unique_tokens, on=ID_COL, how='left')
    for window in [30, 90, 180]:
        util = util.merge(recent_count(window), on=ID_COL, how='left')

    fill_cols = [
        'n_unique_condition_tokens',
        'n_recent_condition_event_days_30d',
        'n_recent_condition_event_days_90d',
        'n_recent_condition_event_days_180d',
    ]
    util[fill_cols] = util[fill_cols].fillna(0)

    util['condition_token_density'] = (
        util['total_condition_tokens'] /
        util['n_condition_event_days'].replace(0, np.nan)
    ).fillna(0)

    util['condition_observation_span_days'] = (
        util['last_condition_day'] - util['first_condition_day']
    ).clip(lower=0)

    # Recency concentration: proportion of event-days occurring near surgery.
    util['recent_30d_fraction'] = (
        util['n_recent_condition_event_days_30d'] /
        util['n_condition_event_days'].replace(0, np.nan)
    ).fillna(0)
    util['recent_90d_fraction'] = (
        util['n_recent_condition_event_days_90d'] /
        util['n_condition_event_days'].replace(0, np.nan)
    ).fillna(0)

    # Add any patients absent from event_days as zero-utilization rows.
    if patient_ids is not None:
        base = pd.DataFrame({ID_COL: [int(p) for p in patient_ids]})
        y_small = y_df[[ID_COL, LABEL_COL]].drop_duplicates(ID_COL)
        base = base.merge(y_small, on=ID_COL, how='left')
        cohort_small = event_days[[ID_COL, 'cohort_name']].drop_duplicates(ID_COL)
        base = base.merge(cohort_small, on=ID_COL, how='left')
        util = base.merge(util.drop(columns=[LABEL_COL, 'cohort_name'], errors='ignore'), on=ID_COL, how='left')

    raw_count_cols = [
        'n_condition_event_days',
        'total_condition_tokens',
        'n_unique_condition_tokens',
        'n_recent_condition_event_days_30d',
        'n_recent_condition_event_days_90d',
        'n_recent_condition_event_days_180d',
        'condition_token_density',
        'condition_observation_span_days',
    ]
    raw_fraction_cols = ['recent_30d_fraction', 'recent_90d_fraction']

    for c in raw_count_cols:
        util[c] = pd.to_numeric(util[c], errors='coerce').fillna(0)
        util[f'util_log1p::{c}'] = np.log1p(util[c])

    for c in raw_fraction_cols:
        util[c] = pd.to_numeric(util[c], errors='coerce').fillna(0).clip(0, 1)
        util[f'util::{c}'] = util[c]

    # Keep timing features on a comparable scale; more negative first day = longer history.
    for c in ['first_condition_day', 'last_condition_day']:
        util[c] = pd.to_numeric(util[c], errors='coerce').fillna(MIN_DAY)
        util[f'util_scaled::{c}'] = util[c] / abs(MIN_DAY)

    return util


def build_utilization_dicts(util_df):
    util_cols = [
        c for c in util_df.columns
        if c.startswith('util_log1p::') or c.startswith('util::') or c.startswith('util_scaled::')
    ]
    lookup = util_df.set_index(ID_COL)[util_cols].to_dict(orient='index') if len(util_df) else {}
    rows = []
    for pid in all_patient_ids:
        rec = {ID_COL: int(pid)}
        vals = lookup.get(int(pid), {})
        for c in util_cols:
            rec[c] = float(vals.get(c, 0.0)) if pd.notna(vals.get(c, 0.0)) else 0.0
        rows.append(rec)
    return rows


utilization_features = build_utilization_features(hybrid_condition_days, token_col=TOKEN_COL, patient_ids=all_patient_ids)
utilization_dicts = build_utilization_dicts(utilization_features)

print('Utilization/documentation-intensity features:', utilization_features.shape)
display(utilization_features.head())

print('Utilization features by outcome:')
summary_cols = [
    'n_condition_event_days',
    'total_condition_tokens',
    'n_unique_condition_tokens',
    'n_recent_condition_event_days_30d',
    'n_recent_condition_event_days_90d',
    'condition_token_density',
    'recent_30d_fraction',
    'recent_90d_fraction',
]
display(utilization_features.groupby(LABEL_COL)[summary_cols].describe().round(2))

plt.figure(figsize=(8, 5))
for lab, label_name in [(0, 'Control'), (1, 'Case')]:
    vals = utilization_features[utilization_features[LABEL_COL] == lab]['n_condition_event_days']
    plt.hist(vals, bins=30, alpha=0.55, label=label_name)
plt.xlabel('Number of condition documentation event-days per patient')
plt.ylabel('Number of patients')
plt.title('Condition documentation intensity by outcome')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
for lab, label_name in [(0, 'Control'), (1, 'Case')]:
    vals = utilization_features[utilization_features[LABEL_COL] == lab]['n_recent_condition_event_days_90d']
    plt.hist(vals, bins=30, alpha=0.55, label=label_name)
plt.xlabel('Condition documentation event-days in final 90 days before surgery')
plt.ylabel('Number of patients')
plt.title('Recent preoperative documentation intensity by outcome')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# ---------------- Feature builders ----------------
# fail-fast guard: rebuild event-day tables, cohort, and sequence maps from source objects
# immediately before feature construction. This prevents stale notebook variables from silently
# reducing n (for example, from 715 to 505).
if 'SOURCE_PATIENT_IDS_AFTER_FILTERS' not in globals() or 'token_df' not in globals():
    raise RuntimeError('Source cohort/token_df missing. Restart kernel and Run All from the top; do not run lower cells only.')

# Force recollapse from the freshly mapped token table, not from any previously saved/stale event-day object.
hybrid_condition_days = collapse_event_days(token_df)
mapped_phecode_days = collapse_event_days(mapped_only_df)
_source_ids = set(SOURCE_PATIENT_IDS_AFTER_FILTERS)
_hybrid_ids = set(hybrid_condition_days[ID_COL].dropna().astype(int))
if _hybrid_ids != _source_ids:
    missing = sorted(_source_ids - _hybrid_ids)[:20]
    raise AssertionError(
        f'Before feature construction, hybrid_condition_days has {len(_hybrid_ids)} patients but '
        f'the post-filter source cohort has {len(_source_ids)}. Missing examples: {missing}'
    )
print('V8 feature-build source-cohort check passed:', len(_hybrid_ids), 'patients')
reset_cohort_from_hybrid_event_days(verbose=True)

def build_frequency_dicts(seq_map, prefix):
    rows = []
    for pid in all_patient_ids:
        cnt = Counter()
        for _, toks in seq_map.get(int(pid), []):
            for t in sorted(set(toks)):
                cnt[f'{prefix}::{t}'] += 1
        d = {ID_COL: int(pid)}
        for k, v in cnt.items():
            d[k] = math.log1p(v)
        rows.append(d)
    return rows


def stdp_weight(delta_days, tau=1.0, lag=3.0, mode='forward'):
    """Return exponential timing weight for a candidate pair.

    forward: event A must precede event B, 0 < delta <= lag.
    same_or_forward: allows same-day pairs, 0 <= delta <= lag.
    bidirectional: symmetric proximity, abs(delta) <= lag. This is not a causal edge.
    """
    if pd.isna(delta_days):
        return 0.0
    d = float(delta_days)
    if mode == 'forward':
        if d <= 0 or d > lag:
            return 0.0
        return math.exp(-d / float(tau))
    if mode == 'same_or_forward':
        if d < 0 or d > lag:
            return 0.0
        return math.exp(-d / float(tau))
    if mode == 'bidirectional':
        ad = abs(d)
        if ad > lag:
            return 0.0
        return math.exp(-ad / float(tau))
    raise ValueError(f'Unsupported transition mode: {mode}')


def strip_domain_prefix(x):
    s = str(x)
    return s.split('::', 1)[1] if s.startswith('domain::') else s


def edge_allowed(a, b, allowed_edges=None):
    if allowed_edges is None:
        return True
    aa = strip_domain_prefix(a)
    bb = strip_domain_prefix(b)
    return bb in allowed_edges.get(aa, [])


def build_transition_dicts(seq_map, lag=7, tau=3, mode='forward', prefix='transition', allowed_edges=None, surprise=False):
    """Build patient-level transition/proximity features.

    For forward/same_or_forward modes, feature names are directional A->B and preserve chronology.
    For bidirectional mode, feature names are symmetric A<->B, using unordered token pairs and
    absolute day gap. These are local proximity/co-documentation features and should not be
    interpreted as causal or disease-progression edges.
    """
    rows = []
    for pid in all_patient_ids:
        events = []
        for day, toks in seq_map.get(int(pid), []):
            for t in sorted(set(toks)):
                events.append((int(day), str(t)))
        events.sort(key=lambda x: (x[0], x[1]))
        token_counts = Counter([t for _, t in events])
        cnt = Counter()
        n = len(events)
        for i in range(n):
            d1, a = events[i]
            if mode == 'bidirectional':
                j_iter = range(i + 1, n)  # unordered pairs only once
            else:
                j_iter = range(n)
            for j in j_iter:
                if i == j:
                    continue
                d2, b = events[j]
                if a == b:
                    continue
                if mode == 'bidirectional':
                    # Symmetric local proximity feature. Do not apply directional edge constraints.
                    left, right = sorted([a, b])
                    feat = f'{prefix}_{mode}_lag{lag}_tau{tau}::{left}<->{right}'
                else:
                    if not edge_allowed(a, b, allowed_edges=allowed_edges):
                        continue
                    feat = f'{prefix}_{mode}_lag{lag}_tau{tau}::{a}->{b}'
                w = stdp_weight(d2 - d1, tau=tau, lag=lag, mode=mode)
                if w <= 0:
                    continue
                if surprise:
                    w = w / math.sqrt(max(token_counts[a], 1) * max(token_counts[b], 1))
                cnt[feat] += w
        d = {ID_COL: int(pid)}
        for k, v in cnt.items():
            d[k] = math.log1p(v)
        rows.append(d)
    return rows


def merge_feature_dicts(*feature_sets):
    merged = []
    for i, pid in enumerate(all_patient_ids):
        d = {ID_COL: int(pid)}
        for fs in feature_sets:
            if not fs:
                continue
            rec = fs[i]
            for k, v in rec.items():
                if k != ID_COL:
                    d[k] = v
        merged.append(d)
    return merged


raw_condition_frequency = build_frequency_dicts(v1_raw_seq, prefix='raw_condition_freq') if len(v1_condition_days) else []
hybrid_frequency = build_frequency_dicts(hybrid_seq, prefix='hybrid_condition_freq')
mapped_phecode_frequency = build_frequency_dicts(mapped_seq, prefix='mapped_phecode_freq')
phegroup_frequency = build_frequency_dicts(group_seq, prefix='phegroup_freq')
domain_frequency = build_frequency_dicts(domain_seq, prefix='domain_freq') if len(domain_days) else []

print('Hybrid frequency example:')
display(pd.DataFrame(hybrid_frequency).head())
print('Mapped Phecode frequency example:')
display(pd.DataFrame(mapped_phecode_frequency).head())


In [ ]:
# ---------------- patient-retention guard before model fitting ----------------
# This cell verifies that all dictionary-based feature sets are generated for every patient in y_df.

def audit_feature_dict_patient_retention(feature_dicts, feature_name):
    if feature_dicts is None:
        print(feature_name, 'is None')
        return
    ids = [int(d.get(ID_COL)) for d in feature_dicts if ID_COL in d]
    id_set = set(ids)
    print(
        feature_name,
        '| rows:', len(feature_dicts),
        '| unique patients:', len(id_set),
        '| y_missing_from_features:', len(base_patient_set - id_set),
        '| feature_extra_not_in_y:', len(id_set - base_patient_set),
    )

# Run this after the feature-builder cell has been executed.
for _name in [
    'hybrid_frequency',
    'mapped_phecode_frequency',
    'phegroup_frequency',
    'domain_frequency',
    'utilization_dicts',
]:
    if _name in globals():
        audit_feature_dict_patient_retention(globals()[_name], _name)


In [ ]:
# ---------------- Vectorization and model evaluation helpers ----------------
def dicts_to_frame(dict_rows):
    if not dict_rows:
        return pd.DataFrame({ID_COL: all_patient_ids})
    df = pd.DataFrame(dict_rows).fillna(0)
    if ID_COL not in df.columns:
        df[ID_COL] = all_patient_ids
    return df


def align_X_y(dict_rows):
    X_df = dicts_to_frame(dict_rows)
    data = y_df[[ID_COL, LABEL_COL]].merge(X_df, on=ID_COL, how='left').fillna(0)
    X = data.drop(columns=[ID_COL, LABEL_COL])
    y = data[LABEL_COL].astype(int).values
    pids = data[ID_COL].astype(int).values
    return X, y, pids


def choose_top_features(X_train, y_train, max_features=1000, min_patient_freq=MIN_PATIENT_FREQ):
    if X_train.shape[1] == 0:
        return []
    prevalence = (X_train > 0).sum(axis=0)
    eligible = prevalence[prevalence >= min_patient_freq].index.tolist()
    if not eligible:
        eligible = prevalence.sort_values(ascending=False).head(min(max_features, len(prevalence))).index.tolist()
    X_el = X_train[eligible]
    if len(eligible) <= max_features:
        return eligible
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        scores, _ = f_classif(X_el.values, y_train)
    scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)
    order = np.argsort(scores)[::-1][:max_features]
    return [eligible[i] for i in order]


def make_model(model_family='ridge'):
    if model_family == 'ridge':
        clf = LogisticRegression(
            penalty='l2', C=1.0, solver='liblinear', class_weight='balanced',
            random_state=RANDOM_STATE, max_iter=5000
        )
    elif model_family == 'elasticnet':
        clf = LogisticRegression(
            penalty='elasticnet', C=0.5, l1_ratio=0.15, solver='saga', class_weight='balanced',
            random_state=RANDOM_STATE, max_iter=5000, n_jobs=-1
        )
    else:
        raise ValueError(model_family)
    return Pipeline([
        ('impute', SimpleImputer(strategy='constant', fill_value=0)),
        ('scale', StandardScaler(with_mean=False)),
        ('clf', clf),
    ])


def partial_auc_specificity(y_true, y_score, spec_min=0.80):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    max_fpr = 1.0 - spec_min
    mask = fpr <= max_fpr
    if mask.sum() < 2:
        return np.nan
    return auc(fpr[mask], tpr[mask]) / max_fpr


def threshold_at_specificity(y_true, y_score, target_spec=0.90):
    fpr, tpr, thr = roc_curve(y_true, y_score)
    spec = 1 - fpr
    ok = np.where(spec >= target_spec)[0]
    if len(ok) == 0:
        return np.nan, np.nan, np.nan
    best = ok[np.argmax(tpr[ok])]
    return float(thr[best]), float(tpr[best]), float(spec[best])


def summarize_predictions(y_true, y_score, model_name):
    out = {
        'model': model_name,
        'n': int(len(y_true)),
        'case_rate': float(np.mean(y_true)),
        'auroc': float(roc_auc_score(y_true, y_score)) if len(np.unique(y_true)) > 1 else np.nan,
        'auprc': float(average_precision_score(y_true, y_score)),
        'brier': float(brier_score_loss(y_true, y_score)),
        'pauc_spec80_100': float(partial_auc_specificity(y_true, y_score, 0.80)),
        'pauc_spec90_100': float(partial_auc_specificity(y_true, y_score, 0.90)),
    }
    for spec in SPEC_TARGETS:
        thr, sens, actual_spec = threshold_at_specificity(y_true, y_score, spec)
        tag = f'spec{int(spec*100)}'
        out[f'{tag}_threshold'] = thr
        out[f'{tag}_sensitivity'] = sens
        out[f'{tag}_specificity'] = actual_spec
    return out


def run_feature_model(model_name, dict_rows, top_k=1000, model_family='ridge'):
    X, y, pids = align_X_y(dict_rows)
    if X.shape[1] == 0:
        raise ValueError(f'{model_name} has zero features.')
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(y), dtype=float)
    coef_rows = []
    fold_feature_counts = []

    for fold, (tr, te) in enumerate(skf.split(X, y), start=1):
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr = y[tr]
        selected = choose_top_features(X_tr, y_tr, max_features=top_k)
        fold_feature_counts.append(len(selected))
        if not selected:
            oof[te] = y_tr.mean()
            continue
        model = make_model(model_family)
        model.fit(X_tr[selected], y_tr)
        oof[te] = model.predict_proba(X_te[selected])[:, 1]

        clf = model.named_steps['clf']
        coefs = clf.coef_.ravel()
        top_idx = np.argsort(np.abs(coefs))[::-1][:50]
        for rank, idx in enumerate(top_idx, start=1):
            coef_rows.append({
                'model': model_name,
                'fold': fold,
                'rank_abs_coef': rank,
                'feature': selected[idx],
                'coef': float(coefs[idx]),
                'abs_coef': float(abs(coefs[idx])),
            })

    metrics = summarize_predictions(y, oof, model_name)
    metrics['model_family'] = model_family
    metrics['n_features_median_fold'] = float(np.median(fold_feature_counts)) if fold_feature_counts else 0
    preds = pd.DataFrame({ID_COL: pids, LABEL_COL: y, f'p_{model_name}': oof})
    coef_df = pd.DataFrame(coef_rows)
    return metrics, preds, coef_df


In [ ]:
# ---------------- Run forward-vs-bidirectional sensitivity feature/model comparisons ----------------

# V7 fail-fast checks before model specification.
if 'SOURCE_PATIENT_IDS_AFTER_FILTERS' not in globals():
    raise RuntimeError('SOURCE_PATIENT_IDS_AFTER_FILTERS is missing. Restart kernel and Run All from the top.')
expected_n = len(SOURCE_PATIENT_IDS_AFTER_FILTERS)
actual_hybrid_n = hybrid_condition_days[ID_COL].nunique() if 'hybrid_condition_days' in globals() else None
actual_y_n = y_df[ID_COL].nunique() if 'y_df' in globals() else None
if actual_hybrid_n != expected_n or actual_y_n != expected_n:
    raise AssertionError(
        f'Before model fitting, source cohort has {expected_n} patients, '
        f'hybrid_condition_days has {actual_hybrid_n}, and y_df has {actual_y_n}. '
        'This indicates stale state or dropped patients. Use Kernel -> Restart & Run All.'
    )

for _required_name in ['hybrid_frequency', 'mapped_phecode_frequency', 'phegroup_frequency', 'utilization_dicts']:
    if _required_name not in globals():
        raise NameError(f'Missing {_required_name}. Run the Feature builders and Utilization feature cells before model fitting.')
    _ids = {int(d.get(ID_COL)) for d in globals()[_required_name] if ID_COL in d}
    if len(_ids) != expected_n:
        raise AssertionError(
            f'{_required_name} has {len(_ids)} patients but expected {expected_n}. '
            'This usually means feature dictionaries were built with stale all_patient_ids. Re-run the Feature builders cell.'
        )

print('V8 pre-model cohort check passed:', expected_n, 'patients')

feature_sets = {}
model_specs = []

if raw_condition_frequency:
    feature_sets['v1_raw_condition_frequency'] = raw_condition_frequency
    model_specs.append(('v1_raw_condition_frequency_ridge', raw_condition_frequency, TOP_K_RAW_FREQ, 'ridge'))

feature_sets['hybrid_frequency'] = hybrid_frequency
model_specs.append(('hybrid_frequency_ridge', hybrid_frequency, TOP_K_HYBRID_FREQ, 'ridge'))

feature_sets['mapped_phecode_frequency'] = mapped_phecode_frequency
model_specs.append(('mapped_phecode_frequency_ridge', mapped_phecode_frequency, TOP_K_HYBRID_FREQ, 'ridge'))

feature_sets['phegroup_frequency'] = phegroup_frequency
model_specs.append(('phegroup_frequency_ridge', phegroup_frequency, TOP_K_PHEGROUP_FREQ, 'ridge'))

if domain_frequency:
    feature_sets['domain_frequency'] = domain_frequency
    model_specs.append(('domain_frequency_ridge', domain_frequency, TOP_K_DOMAIN_FREQ, 'ridge'))

# Utilization / documentation-intensity adjustment models.
feature_sets['utilization_only'] = utilization_dicts
model_specs.append(('utilization_only_ridge', utilization_dicts, TOP_K_UTILIZATION, 'ridge'))

hybrid_plus_util = merge_feature_dicts(hybrid_frequency, utilization_dicts)
feature_sets['hybrid_frequency_plus_utilization'] = hybrid_plus_util
model_specs.append(('hybrid_frequency_plus_utilization_ridge', hybrid_plus_util, TOP_K_COMBINED_UTIL, 'ridge'))

mapped_plus_util = merge_feature_dicts(mapped_phecode_frequency, utilization_dicts)
feature_sets['mapped_phecode_frequency_plus_utilization'] = mapped_plus_util
model_specs.append(('mapped_phecode_frequency_plus_utilization_ridge', mapped_plus_util, TOP_K_COMBINED_UTIL, 'ridge'))

phegroup_plus_util = merge_feature_dicts(phegroup_frequency, utilization_dicts)
feature_sets['phegroup_frequency_plus_utilization'] = phegroup_plus_util
model_specs.append(('phegroup_frequency_plus_utilization_ridge', phegroup_plus_util, TOP_K_COMBINED_UTIL, 'ridge'))

if domain_frequency:
    domain_plus_util = merge_feature_dicts(domain_frequency, utilization_dicts)
    feature_sets['domain_frequency_plus_utilization'] = domain_plus_util
    model_specs.append(('domain_frequency_plus_utilization_ridge', domain_plus_util, TOP_K_COMBINED_UTIL, 'ridge'))

for lag, tau in TRANSITION_GRID:
    for mode in TRANSITION_MODES:
        # Biologically interpretable official mapped Phecode transitions.
        map_key = f'mapped_phecode_transition_{mode}_lag{lag}_tau{tau}'
        map_tr = build_transition_dicts(
            mapped_transition_seq,
            lag=lag,
            tau=tau,
            mode=mode,
            prefix='mapped_phecode_transition',
            allowed_edges=None,
            surprise=False,
        )
        feature_sets[map_key] = map_tr
        model_specs.append((map_key + '_ridge', map_tr, TOP_K_MAPPED_TRANSITION, 'ridge'))

        combo_key = 'hybrid_freq_plus_' + map_key
        combo = merge_feature_dicts(hybrid_frequency, map_tr)
        feature_sets[combo_key] = combo
        model_specs.append((combo_key + '_ridge', combo, TOP_K_COMBINED, 'ridge'))

        combo_util_key = combo_key + '_plus_utilization'
        combo_util = merge_feature_dicts(hybrid_frequency, map_tr, utilization_dicts)
        feature_sets[combo_util_key] = combo_util
        model_specs.append((combo_util_key + '_ridge', combo_util, TOP_K_COMBINED_UTIL, 'ridge'))

        group_key = f'phegroup_transition_{mode}_lag{lag}_tau{tau}'
        group_tr = build_transition_dicts(
            group_transition_seq,
            lag=lag,
            tau=tau,
            mode=mode,
            prefix='phegroup_transition',
            allowed_edges=None,
            surprise=False,
        )
        feature_sets[group_key] = group_tr
        model_specs.append((group_key + '_ridge', group_tr, TOP_K_PHEGROUP_FREQ, 'ridge'))

        group_combo_key = 'hybrid_freq_plus_' + group_key
        group_combo = merge_feature_dicts(hybrid_frequency, group_tr)
        feature_sets[group_combo_key] = group_combo
        model_specs.append((group_combo_key + '_ridge', group_combo, TOP_K_COMBINED, 'ridge'))

        group_combo_util_key = group_combo_key + '_plus_utilization'
        group_combo_util = merge_feature_dicts(hybrid_frequency, group_tr, utilization_dicts)
        feature_sets[group_combo_util_key] = group_combo_util
        model_specs.append((group_combo_util_key + '_ridge', group_combo_util, TOP_K_COMBINED_UTIL, 'ridge'))

        if domain_frequency:
            if mode == 'bidirectional':
                dom_key = f'domain_proximity_{mode}_lag{lag}_tau{tau}'
                dom_prefix = 'domain_proximity'
                dom_allowed_edges = None
            else:
                dom_key = f'constrained_domain_transition_{mode}_lag{lag}_tau{tau}'
                dom_prefix = 'constrained_domain_transition'
                dom_allowed_edges = PLAUSIBLE_DOMAIN_TRANSITIONS
            dom_tr = build_transition_dicts(
                domain_transition_seq,
                lag=lag,
                tau=tau,
                mode=mode,
                prefix=dom_prefix,
                allowed_edges=dom_allowed_edges,
                surprise=False,
            )
            feature_sets[dom_key] = dom_tr
            model_specs.append((dom_key + '_ridge', dom_tr, TOP_K_DOMAIN_TRANSITION, 'ridge'))

            dom_combo_key = 'hybrid_freq_plus_' + dom_key
            dom_combo = merge_feature_dicts(hybrid_frequency, dom_tr)
            feature_sets[dom_combo_key] = dom_combo
            model_specs.append((dom_combo_key + '_ridge', dom_combo, TOP_K_COMBINED, 'ridge'))

            dom_combo_util_key = dom_combo_key + '_plus_utilization'
            dom_combo_util = merge_feature_dicts(hybrid_frequency, dom_tr, utilization_dicts)
            feature_sets[dom_combo_util_key] = dom_combo_util
            model_specs.append((dom_combo_util_key + '_ridge', dom_combo_util, TOP_K_COMBINED_UTIL, 'ridge'))

print('Model specs to run:', len(model_specs))

metrics_rows = []
preds_df = y_df[[ID_COL, LABEL_COL]].copy().reset_index(drop=True)
coef_parts = []

for i, (name, dicts, top_k, family) in enumerate(model_specs, start=1):
    print(f'[{i}/{len(model_specs)}] {name}')
    try:
        m, p, c = run_feature_model(name, dicts, top_k=top_k, model_family=family)
        metrics_rows.append(m)
        preds_df = preds_df.merge(p[[ID_COL, f'p_{name}']], on=ID_COL, how='left')
        if len(c):
            coef_parts.append(c)
    except Exception as e:
        print('FAILED:', name, e)

metrics_df = pd.DataFrame(metrics_rows).sort_values(['auroc', 'auprc'], ascending=False).reset_index(drop=True)
coef_df = pd.concat(coef_parts, ignore_index=True) if coef_parts else pd.DataFrame()

metrics_path = OUT_DIR / 'v8_phecode_hybrid_transition_bidirectional_metrics.csv'
preds_path = OUT_DIR / 'v8_phecode_hybrid_transition_bidirectional_oof_predictions.csv'
coef_path = OUT_DIR / 'v8_phecode_hybrid_transition_bidirectional_top_fold_coefficients.csv'
metrics_df.to_csv(metrics_path, index=False)
preds_df.to_csv(preds_path, index=False)
coef_df.to_csv(coef_path, index=False)

print('Saved:', metrics_path)
print('Saved:', preds_path)
print('Saved:', coef_path)
display(metrics_df.head(30))


In [ ]:
# ---------------- Tuning visualizations ----------------
if len(metrics_df):
    display_cols = ['model', 'n', 'case_rate', 'auroc', 'auprc', 'brier', 'pauc_spec80_100', 'pauc_spec90_100']
    display(metrics_df[display_cols].head(20))

    # Best AUROC by model group.
    tmp = metrics_df.copy()
    tmp['group'] = tmp['model'].str.replace(r'_lag\d+_tau\d+_ridge$', '', regex=True)
    best_by_group = tmp.sort_values('auroc', ascending=False).groupby('group', as_index=False).head(1)
    best_by_group = best_by_group.sort_values('auroc', ascending=True)

    plt.figure(figsize=(10, max(5, 0.3 * len(best_by_group))))
    plt.barh(best_by_group['group'], best_by_group['auroc'])
    plt.xlabel('OOF AUROC')
    plt.title('Best V8 model per feature family')
    plt.tight_layout()
    plt.show()

    # Heatmap-like tables for lag/tau models.
    for prefix in [
        'hybrid_freq_plus_mapped_phecode_transition_forward',
        'hybrid_freq_plus_mapped_phecode_transition_bidirectional',
        'hybrid_freq_plus_phegroup_transition_forward',
        'hybrid_freq_plus_phegroup_transition_bidirectional',
        'hybrid_freq_plus_domain_proximity_bidirectional',
    ]:
        sub = metrics_df[metrics_df['model'].str.contains(prefix, regex=False)].copy()
        if not len(sub):
            continue
        sub['lag'] = sub['model'].str.extract(r'lag(\d+)').astype(float)
        sub['tau'] = sub['model'].str.extract(r'tau(\d+)').astype(float)
        piv = sub.pivot_table(index='lag', columns='tau', values='auroc', aggfunc='max')
        print(prefix)
        display(piv.round(4))
        plt.figure(figsize=(7, 4))
        plt.imshow(piv.values, aspect='auto')
        plt.colorbar(label='AUROC')
        plt.xticks(np.arange(len(piv.columns)), [str(int(c)) for c in piv.columns])
        plt.yticks(np.arange(len(piv.index)), [str(int(i)) for i in piv.index])
        plt.xlabel('tau')
        plt.ylabel('lag')
        plt.title(prefix)
        plt.tight_layout()
        plt.show()


In [ ]:
# ---------------- Forward vs bidirectional sensitivity summary ----------------
if len(metrics_df):
    sens = metrics_df.copy()
    sens['encoding'] = np.where(sens['model'].str.contains('bidirectional', regex=False), 'bidirectional/proximity',
                          np.where(sens['model'].str.contains('forward', regex=False), 'forward/ordered', 'non-transition'))
    sens['transition_level'] = np.select(
        [sens['model'].str.contains('phegroup_transition', regex=False),
         sens['model'].str.contains('mapped_phecode_transition', regex=False),
         sens['model'].str.contains('domain_proximity|constrained_domain_transition', regex=True)],
        ['phegroup', 'mapped_phecode', 'domain'],
        default='baseline'
    )
    best_sens = (sens.sort_values(['transition_level','encoding','auroc'], ascending=[True, True, False])
                   .groupby(['transition_level','encoding'], as_index=False)
                   .head(1)
                   .sort_values(['transition_level','encoding']))
    print('Best model within each transition level and encoding:')
    display(best_sens[['transition_level','encoding','model','n','auroc','auprc','brier','n_features_median_fold']])

    # Paired comparison table for forward vs bidirectional within same lag/tau family.
    trans = sens[sens['encoding'].isin(['forward/ordered','bidirectional/proximity'])].copy()
    trans['base_family'] = trans['model'].str.replace('bidirectional', 'MODE', regex=False).str.replace('forward', 'MODE', regex=False)
    trans['lag'] = trans['model'].str.extract(r'lag(\d+)')
    trans['tau'] = trans['model'].str.extract(r'tau(\d+)')
    comp = trans.pivot_table(index=['transition_level','base_family','lag','tau'], columns='encoding', values='auroc', aggfunc='max').reset_index()
    if {'forward/ordered','bidirectional/proximity'}.issubset(set(comp.columns)):
        comp['delta_bidirectional_minus_forward'] = comp['bidirectional/proximity'] - comp['forward/ordered']
        print('AUROC delta: bidirectional proximity minus forward ordered, matched by lag/tau family')
        display(comp.sort_values('delta_bidirectional_minus_forward', ascending=False).head(30))


In [ ]:
# ---------------- ROC and PR curves for representative models ----------------
if len(metrics_df):
    top_models = metrics_df['model'].head(5).tolist()
    baseline_candidates = [m for m in ['hybrid_frequency_ridge', 'mapped_phecode_frequency_ridge', 'phegroup_frequency_ridge'] if m in metrics_df['model'].values]
    plot_models = list(dict.fromkeys(baseline_candidates + top_models))[:8]

    y_true = preds_df[LABEL_COL].values

    plt.figure(figsize=(7, 6))
    for m in plot_models:
        col = 'p_' + m
        if col not in preds_df.columns:
            continue
        y_score = preds_df[col].values
        fpr, tpr, _ = roc_curve(y_true, y_score)
        plt.plot(fpr, tpr, label=f'{m} AUC={roc_auc_score(y_true, y_score):.3f}')
    plt.plot([0, 1], [0, 1], linestyle='--', linewidth=1)
    plt.xlabel('False positive rate')
    plt.ylabel('True positive rate')
    plt.title('ROC curves: V3 representative models')
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(7, 6))
    for m in plot_models:
        col = 'p_' + m
        if col not in preds_df.columns:
            continue
        y_score = preds_df[col].values
        prec, rec, _ = precision_recall_curve(y_true, y_score)
        plt.plot(rec, prec, label=f'{m} AP={average_precision_score(y_true, y_score):.3f}')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-recall curves: V3 representative models')
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


In [ ]:
# ---------------- Transition interpretation audit ----------------
if len(coef_df):
    transition_coef = coef_df[coef_df['feature'].str.contains('transition', case=False, na=False)].copy()
    if len(transition_coef):
        edge_summary = (
            transition_coef
            .groupby(['model', 'feature'], as_index=False)
            .agg(mean_coef=('coef', 'mean'), mean_abs_coef=('abs_coef', 'mean'), n_folds=('fold', 'nunique'))
            .sort_values(['mean_abs_coef', 'n_folds'], ascending=False)
        )
        edge_summary_path = OUT_DIR / 'v3_transition_edge_coefficient_audit.csv'
        edge_summary.to_csv(edge_summary_path, index=False)
        print('Saved:', edge_summary_path)
        display(edge_summary.head(50))
    else:
        print('No transition coefficients found.')

    # Top hybrid frequency features, useful to see whether unmapped fallback dominates.
    freq_coef = coef_df[coef_df['feature'].str.contains('freq', case=False, na=False)].copy()
    if len(freq_coef):
        freq_summary = (
            freq_coef.groupby(['model', 'feature'], as_index=False)
            .agg(mean_coef=('coef', 'mean'), mean_abs_coef=('abs_coef', 'mean'), n_folds=('fold', 'nunique'))
            .sort_values(['mean_abs_coef', 'n_folds'], ascending=False)
        )
        freq_path = OUT_DIR / 'v3_frequency_feature_coefficient_audit.csv'
        freq_summary.to_csv(freq_path, index=False)
        print('Saved:', freq_path)
        display(freq_summary.head(50))


In [ ]:
## prediction-head tuning

This section freezes the feature blocks and compares prediction heads under patient-level outer cross-validation. Hyperparameters are selected only within each training fold. Tree-based heads are included only if the required packages are available.


In [ ]:

# ---------------- freeze feature blocks for prediction-head tuning ----------------

if 'metrics_df' not in globals() or len(metrics_df) == 0:
    raise RuntimeError('metrics_df is missing. Run the V8 feature-comparison cell first.')
if 'feature_sets' not in globals() or len(feature_sets) == 0:
    raise RuntimeError('feature_sets is missing. Run the V8 feature-comparison cell first.')

# Cohort check: V9 must use the source-preserved cohort.
expected_n = len(SOURCE_PATIENT_IDS_AFTER_FILTERS)
if y_df[ID_COL].nunique() != expected_n:
    raise AssertionError(f'V9 cohort check failed: y_df has {y_df[ID_COL].nunique()} but expected {expected_n}.')
print('V9 cohort check passed:', expected_n, 'patients')


def best_feature_key_from_metrics(include_terms, exclude_terms=None, require_feature_set=True):
    """Return the best feature_set key from metrics_df based on AUROC/AUPRC."""
    exclude_terms = exclude_terms or []
    sub = metrics_df.copy()
    for term in include_terms:
        sub = sub[sub['model'].str.contains(term, regex=False, na=False)]
    for term in exclude_terms:
        sub = sub[~sub['model'].str.contains(term, regex=False, na=False)]
    if len(sub) == 0:
        return None, None
    sub = sub.sort_values(['auroc', 'auprc'], ascending=False)
    for _, r in sub.iterrows():
        key = str(r['model'])
        if key.endswith('_ridge'):
            key = key[:-len('_ridge')]
        if (not require_feature_set) or key in feature_sets:
            return key, r.to_dict()
    return None, None

frozen_blocks = []

def add_block(block_name, feature_key, top_k, source_note):
    if feature_key is None or feature_key not in feature_sets:
        print('Skipping missing feature block:', block_name, feature_key)
        return
    frozen_blocks.append({
        'block_name': block_name,
        'feature_key': feature_key,
        'dict_rows': feature_sets[feature_key],
        'top_k': int(top_k),
        'source_note': source_note,
    })

# Core burden/utilization blocks.
add_block('mapped_phecode_frequency', 'mapped_phecode_frequency', TOP_K_HYBRID_FREQ, 'official mapped Phecode frequency baseline')
add_block('hybrid_frequency', 'hybrid_frequency', TOP_K_HYBRID_FREQ, 'Phecode plus unmapped fallback frequency baseline')
add_block('hybrid_frequency_plus_utilization', 'hybrid_frequency_plus_utilization', TOP_K_COMBINED_UTIL, 'hybrid frequency plus documentation-intensity covariates')

# Primary biologically interpretable transition block: forward Phegroup.
key, row = best_feature_key_from_metrics(['hybrid_freq_plus_phegroup_transition_forward'], exclude_terms=['plus_utilization'])
add_block('best_forward_phegroup_transition', key, TOP_K_COMBINED, 'best forward-only Phegroup transition block from V8')

key_util, row_util = best_feature_key_from_metrics(['hybrid_freq_plus_phegroup_transition_forward', 'plus_utilization'])
add_block('best_forward_phegroup_transition_plus_utilization', key_util, TOP_K_COMBINED_UTIL, 'best forward-only Phegroup transition plus utilization block from V8')

# Secondary constrained domain block.
key_dom, row_dom = best_feature_key_from_metrics(['hybrid_freq_plus_constrained_domain_transition_forward'], exclude_terms=['plus_utilization'])
add_block('best_forward_constrained_domain_transition', key_dom, TOP_K_COMBINED, 'best forward-only constrained PDAC-domain transition block from V8')

# Optional sensitivity blocks: best bidirectional/proximity versions.
key_bi_group, row_bi_group = best_feature_key_from_metrics(['hybrid_freq_plus_phegroup_transition_bidirectional'], exclude_terms=['plus_utilization'])
add_block('best_bidirectional_phegroup_proximity', key_bi_group, TOP_K_COMBINED, 'best bidirectional Phegroup proximity sensitivity block from V8')

key_bi_dom, row_bi_dom = best_feature_key_from_metrics(['hybrid_freq_plus_domain_proximity_bidirectional'], exclude_terms=['plus_utilization'])
add_block('best_bidirectional_domain_proximity', key_bi_dom, TOP_K_COMBINED, 'best bidirectional domain proximity sensitivity block from V8')

frozen_block_summary = pd.DataFrame([
    {
        'block_name': b['block_name'],
        'feature_key': b['feature_key'],
        'top_k': b['top_k'],
        'source_note': b['source_note'],
        'n_patients_in_dicts': len({int(d.get(ID_COL)) for d in b['dict_rows'] if ID_COL in d}),
    }
    for b in frozen_blocks
])
print('Frozen feature blocks for V9 head tuning:')
display(frozen_block_summary)

frozen_block_path = OUT_DIR / 'v9_frozen_feature_blocks.csv'
frozen_block_summary.to_csv(frozen_block_path, index=False)
print('Saved:', frozen_block_path)


In [ ]:

# ---------------- prediction-head definitions ----------------
# Logistic heads use inner CV inside each outer training fold. Tree heads are optional and calibrated.

AVAILABLE_TREE_HEADS = []
try:
    from xgboost import XGBClassifier
    AVAILABLE_TREE_HEADS.append('xgboost_calibrated')
except Exception as e:
    print('XGBoost unavailable; skipping xgboost_calibrated:', e)

try:
    from lightgbm import LGBMClassifier
    AVAILABLE_TREE_HEADS.append('lightgbm_calibrated')
except Exception as e:
    print('LightGBM unavailable; skipping lightgbm_calibrated:', e)

PREDICTION_HEADS = [
    'ridge_cv',
    'lasso_cv',
    'elasticnet_cv',
]
if RUN_TREE_HEADS:
    PREDICTION_HEADS += AVAILABLE_TREE_HEADS

print('Prediction heads to evaluate:', PREDICTION_HEADS)


def make_prediction_head(head_name, y_train):
    """Create a prediction head. Hyperparameter tuning/calibration happens inside outer train fold."""
    y_train = np.asarray(y_train).astype(int)
    inner_cv = max(2, min(INNER_CV_SPLITS, np.bincount(y_train).min()))

    if head_name == 'ridge_cv':
        clf = LogisticRegressionCV(
            Cs=HEAD_TUNING_CS,
            cv=inner_cv,
            penalty='l2',
            solver='liblinear',
            scoring='roc_auc',
            class_weight='balanced',
            random_state=RANDOM_STATE,
            max_iter=5000,
            n_jobs=-1,
            refit=True,
        )
        return Pipeline([
            ('impute', SimpleImputer(strategy='constant', fill_value=0)),
            ('scale', StandardScaler(with_mean=False)),
            ('clf', clf),
        ])

    if head_name == 'lasso_cv':
        clf = LogisticRegressionCV(
            Cs=HEAD_TUNING_CS,
            cv=inner_cv,
            penalty='l1',
            solver='saga',
            scoring='roc_auc',
            class_weight='balanced',
            random_state=RANDOM_STATE,
            max_iter=8000,
            n_jobs=-1,
            refit=True,
        )
        return Pipeline([
            ('impute', SimpleImputer(strategy='constant', fill_value=0)),
            ('scale', StandardScaler(with_mean=False)),
            ('clf', clf),
        ])

    if head_name == 'elasticnet_cv':
        clf = LogisticRegressionCV(
            Cs=HEAD_TUNING_CS,
            cv=inner_cv,
            penalty='elasticnet',
            solver='saga',
            l1_ratios=HEAD_TUNING_L1_RATIOS,
            scoring='roc_auc',
            class_weight='balanced',
            random_state=RANDOM_STATE,
            max_iter=8000,
            n_jobs=-1,
            refit=True,
        )
        return Pipeline([
            ('impute', SimpleImputer(strategy='constant', fill_value=0)),
            ('scale', StandardScaler(with_mean=False)),
            ('clf', clf),
        ])

    if head_name == 'xgboost_calibrated':
        if 'XGBClassifier' not in globals():
            raise RuntimeError('XGBoost is not available in this environment.')
        neg = max(1, int((y_train == 0).sum()))
        pos = max(1, int((y_train == 1).sum()))
        base = XGBClassifier(
            n_estimators=150,
            max_depth=2,
            learning_rate=0.03,
            subsample=0.80,
            colsample_bytree=0.80,
            min_child_weight=5,
            reg_alpha=0.5,
            reg_lambda=5.0,
            objective='binary:logistic',
            eval_metric='logloss',
            scale_pos_weight=neg / pos,
            random_state=RANDOM_STATE,
            n_jobs=4,
        )
        clf = CalibratedClassifierCV(base, cv=inner_cv, method='sigmoid')
        return Pipeline([
            ('impute', SimpleImputer(strategy='constant', fill_value=0)),
            ('clf', clf),
        ])

    if head_name == 'lightgbm_calibrated':
        if 'LGBMClassifier' not in globals():
            raise RuntimeError('LightGBM is not available in this environment.')
        base = LGBMClassifier(
            n_estimators=150,
            max_depth=2,
            learning_rate=0.03,
            subsample=0.80,
            colsample_bytree=0.80,
            min_child_samples=20,
            reg_alpha=0.5,
            reg_lambda=5.0,
            class_weight='balanced',
            random_state=RANDOM_STATE,
            n_jobs=4,
            verbose=-1,
        )
        clf = CalibratedClassifierCV(base, cv=inner_cv, method='sigmoid')
        return Pipeline([
            ('impute', SimpleImputer(strategy='constant', fill_value=0)),
            ('clf', clf),
        ])

    raise ValueError(f'Unknown head_name: {head_name}')


def extract_fold_hyperparams(model, head_name):
    params = {'head': head_name}
    clf = model.named_steps.get('clf')
    if hasattr(clf, 'C_'):
        params['selected_C'] = float(np.ravel(clf.C_)[0])
    if hasattr(clf, 'l1_ratio_') and clf.l1_ratio_ is not None:
        params['selected_l1_ratio'] = float(np.ravel(clf.l1_ratio_)[0])
    return params


def extract_linear_importance(model, selected_features, model_name, fold):
    rows = []
    clf = model.named_steps.get('clf')
    if hasattr(clf, 'coef_'):
        coefs = clf.coef_.ravel()
        top_idx = np.argsort(np.abs(coefs))[::-1][:50]
        for rank, idx in enumerate(top_idx, start=1):
            rows.append({
                'model': model_name,
                'fold': fold,
                'rank_abs_coef': rank,
                'feature': selected_features[idx],
                'importance': float(coefs[idx]),
                'abs_importance': float(abs(coefs[idx])),
                'importance_type': 'coefficient',
            })
    return rows


In [ ]:

# ---------------- run prediction-head tuning on frozen feature blocks ----------------

v9_metrics_rows = []
v9_preds_df = y_df[[ID_COL, LABEL_COL]].copy().reset_index(drop=True)
v9_importance_parts = []
v9_fold_param_rows = []


def run_feature_block_head_model(block_name, feature_key, dict_rows, top_k, head_name):
    X, y, pids = align_X_y(dict_rows)
    if len(y) != expected_n:
        raise AssertionError(f'{block_name}/{head_name}: y has {len(y)} patients; expected {expected_n}.')
    if X.shape[1] == 0:
        raise ValueError(f'{block_name}/{head_name} has zero features.')

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(y), dtype=float)
    importance_rows = []
    fold_params = []
    fold_feature_counts = []

    for fold, (tr, te) in enumerate(skf.split(X, y), start=1):
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr = y[tr]
        selected = choose_top_features(X_tr, y_tr, max_features=top_k)
        fold_feature_counts.append(len(selected))
        if not selected:
            oof[te] = y_tr.mean()
            continue

        model = make_prediction_head(head_name, y_tr)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            model.fit(X_tr[selected], y_tr)
        oof[te] = model.predict_proba(X_te[selected])[:, 1]

        params = extract_fold_hyperparams(model, head_name)
        params.update({
            'block_name': block_name,
            'feature_key': feature_key,
            'model': f'{block_name}__{head_name}',
            'fold': fold,
            'n_selected_features': len(selected),
            'outer_train_cases': int(y_tr.sum()),
            'outer_train_controls': int((y_tr == 0).sum()),
        })
        fold_params.append(params)

        importance_rows.extend(
            extract_linear_importance(model, selected, f'{block_name}__{head_name}', fold)
        )

    model_name = f'{block_name}__{head_name}'
    metrics = summarize_predictions(y, oof, model_name)
    metrics.update({
        'block_name': block_name,
        'feature_key': feature_key,
        'prediction_head': head_name,
        'n_features_median_fold': float(np.median(fold_feature_counts)) if fold_feature_counts else 0,
        'vv_uq_role': 'prediction_head_tuning_on_frozen_feature_block',
    })
    preds = pd.DataFrame({ID_COL: pids, LABEL_COL: y, f'p_{model_name}': oof})
    return metrics, preds, pd.DataFrame(importance_rows), pd.DataFrame(fold_params)

print('Frozen blocks:', len(frozen_blocks))
print('Prediction heads:', PREDICTION_HEADS)
print('Total V9 runs:', len(frozen_blocks) * len(PREDICTION_HEADS))

run_idx = 0
for b in frozen_blocks:
    for head in PREDICTION_HEADS:
        run_idx += 1
        print(f'[{run_idx}/{len(frozen_blocks) * len(PREDICTION_HEADS)}] {b["block_name"]} :: {head}')
        try:
            m, p, imp, hp = run_feature_block_head_model(
                block_name=b['block_name'],
                feature_key=b['feature_key'],
                dict_rows=b['dict_rows'],
                top_k=b['top_k'],
                head_name=head,
            )
            v9_metrics_rows.append(m)
            v9_preds_df = v9_preds_df.merge(p[[ID_COL, f'p_{m["model"]}']], on=ID_COL, how='left')
            if len(imp):
                v9_importance_parts.append(imp)
            if len(hp):
                v9_fold_param_rows.append(hp)
        except Exception as e:
            print('FAILED:', b['block_name'], head, e)
            v9_metrics_rows.append({
                'model': f'{b["block_name"]}__{head}',
                'block_name': b['block_name'],
                'feature_key': b['feature_key'],
                'prediction_head': head,
                'error': str(e),
            })

v9_metrics_df = pd.DataFrame(v9_metrics_rows)
if 'auroc' in v9_metrics_df.columns:
    v9_metrics_df = v9_metrics_df.sort_values(['auroc', 'auprc'], ascending=False).reset_index(drop=True)

v9_importance_df = pd.concat(v9_importance_parts, ignore_index=True) if v9_importance_parts else pd.DataFrame()
v9_fold_params_df = pd.concat(v9_fold_param_rows, ignore_index=True) if v9_fold_param_rows else pd.DataFrame()

v9_metrics_path = OUT_DIR / 'v9_prediction_head_tuning_metrics.csv'
v9_preds_path = OUT_DIR / 'v9_prediction_head_tuning_oof_predictions.csv'
v9_importance_path = OUT_DIR / 'v9_prediction_head_tuning_feature_importance.csv'
v9_fold_params_path = OUT_DIR / 'v9_prediction_head_tuning_fold_hyperparams.csv'

v9_metrics_df.to_csv(v9_metrics_path, index=False)
v9_preds_df.to_csv(v9_preds_path, index=False)
v9_importance_df.to_csv(v9_importance_path, index=False)
v9_fold_params_df.to_csv(v9_fold_params_path, index=False)

print('Saved:', v9_metrics_path)
print('Saved:', v9_preds_path)
print('Saved:', v9_importance_path)
print('Saved:', v9_fold_params_path)
display(v9_metrics_df.head(30))


In [ ]:

# ---------------- prediction-head tuning summary and deltas ----------------
if 'v9_metrics_df' in globals() and len(v9_metrics_df):
    ok = v9_metrics_df[v9_metrics_df['auroc'].notna()].copy() if 'auroc' in v9_metrics_df.columns else pd.DataFrame()
    if len(ok):
        print('Best prediction head within each frozen feature block:')
        best_by_block = (
            ok.sort_values(['block_name', 'auroc', 'auprc'], ascending=[True, False, False])
              .groupby('block_name', as_index=False)
              .head(1)
              .sort_values('auroc', ascending=False)
        )
        display(best_by_block[['block_name', 'feature_key', 'prediction_head', 'n', 'auroc', 'auprc', 'brier', 'n_features_median_fold']])

        # Delta against ridge_cv for the same frozen feature block.
        wide = ok.pivot_table(index=['block_name','feature_key'], columns='prediction_head', values='auroc', aggfunc='max').reset_index()
        if 'ridge_cv' in wide.columns:
            for col in [c for c in wide.columns if c not in ['block_name','feature_key','ridge_cv']]:
                wide[f'delta_{col}_minus_ridge_cv'] = wide[col] - wide['ridge_cv']
            print('AUROC deltas versus ridge_cv within the same frozen block:')
            display(wide.sort_values('ridge_cv', ascending=False))

        # AUPRC-focused summary, useful in imbalanced recurrence prediction.
        print('Best AUPRC models:')
        display(ok.sort_values(['auprc','auroc'], ascending=False).head(15)[['model','block_name','prediction_head','n','auroc','auprc','brier']])

        best_by_block_path = OUT_DIR / 'v9_best_prediction_head_by_block.csv'
        best_by_block.to_csv(best_by_block_path, index=False)
        print('Saved:', best_by_block_path)


In [ ]:

# ---------------- VUQ notes for README/manuscript ----------------
v9_vvuq_notes = {
    'verification': [
        'Cohort was anchored to SOURCE_PATIENT_IDS_AFTER_FILTERS and required to match y_df before model fitting.',
        'Frozen feature blocks were selected from completed V7/V8 feature-engineering runs; feature construction was not retuned during prediction-head tuning.',
        'Outer folds used patient-level stratified cross-validation; no patient was split across train/test folds.',
    ],
    'validation': [
        'Prediction heads were compared on the same frozen feature blocks and same outer folds.',
        'Regularization hyperparameters were selected only inside outer training folds using inner cross-validation.',
        'Bidirectional/proximity feature blocks, when included, were treated as sensitivity features rather than causal or disease-progression edges.',
    ],
    'uncertainty_quantification_next_steps': [
        'Use paired bootstrap confidence intervals for AUROC/AUPRC/Brier deltas versus ridge_cv on the same OOF predictions.',
        'Use paired DeLong testing for AUROC comparisons when reporting a small number of prespecified model comparisons.',
        'Apply feature-stability filtering before interpreting coefficients; prioritize features appearing across multiple folds.',
    ],
}

v9_vvuq_path = OUT_DIR / 'v9_vvuq_notes.json'
with open(v9_vvuq_path, 'w') as f:
    json.dump(v9_vvuq_notes, f, indent=2)
print('Saved:', v9_vvuq_path)
print(json.dumps(v9_vvuq_notes, indent=2))


In [ ]:
# ---------------- Manifest ----------------
# Public-safe manifest: stores file names and run settings, not absolute local paths.
def _safe_name(x):
    if x is None:
        return None
    try:
        return Path(x).name
    except Exception:
        return str(x)

manifest = {
    'created_at': datetime.now().isoformat(),
    'notebook': 'pdac_phecode_hybrid_condition_transition_v9_prediction_head_tuning.ipynb',
    'purpose': 'Tune prediction heads on frozen, cohort-preserved Phecode-hybrid transition feature blocks under VVUQ; includes V8 bidirectional sensitivity context.',
    'source_files': {k: _safe_name(v) for k, v in source_paths.items()},
    'phecode_map_file': _safe_name(PHECODE_MAP_PATH),
    'phecode_info_file': _safe_name(PHECODE_INFO_PATH),
    'output_directory': _safe_name(OUT_DIR),
    'min_day': MIN_DAY,
    'max_day': MAX_DAY,
    'transition_grid': TRANSITION_GRID,
    'transition_modes': TRANSITION_MODES,
    'use_first_onset_for_transitions': USE_FIRST_ONSET_FOR_TRANSITIONS,
    'interpretation': {
        'hybrid_tokens': 'Phecode where mapped; otherwise unmapped_condition fallback tokens retained for prediction/burden adjustment.',
        'mapped_phecode_transitions': 'Official Phecode-based forward transitions used for primary biologically interpretable analysis; bidirectional versions are sensitivity/proximity features only.',
        'domain_transitions': 'Secondary PDAC-relevant interpretable domain layer; forward constrained edges are not a definitive causal graph; bidirectional domain proximity is exploratory.'
    },
    'privacy_note': (
        'This manifest intentionally stores file names only. Do not commit local outputs containing '
        'patient-level identifiers, predictions, or institution-specific paths.'
    ),
    'outputs': {
        'hybrid_event_days': _safe_name(hybrid_path) if 'hybrid_path' in globals() else None,
        'mapped_event_days': _safe_name(mapped_path) if 'mapped_path' in globals() else None,
        'metrics': _safe_name(metrics_path) if 'metrics_path' in globals() else None,
        'predictions': _safe_name(preds_path) if 'preds_path' in globals() else None,
        'coefficients': _safe_name(coef_path) if 'coef_path' in globals() else None,
        'v9_head_tuning_metrics': _safe_name(v9_metrics_path) if 'v9_metrics_path' in globals() else None,
        'v9_head_tuning_predictions': _safe_name(v9_preds_path) if 'v9_preds_path' in globals() else None,
        'v9_head_tuning_importance': _safe_name(v9_importance_path) if 'v9_importance_path' in globals() else None,
        'v9_head_tuning_fold_hyperparams': _safe_name(v9_params_path) if 'v9_params_path' in globals() else None,
    }
}

manifest_path = OUT_DIR / 'manifest_public_safe.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)
print('Saved public-safe manifest:', manifest_path.name)
manifest
